# Data cleaning

## Column Renames
- `institutions_distinct_count` → `authorship_count`
  - Reason: column counts author-institution pairs, not distinct institutions
  - Date: [today]
  - Action needed on next pull: either rename incoming column or pull true 
    distinct institution count via authorships[].institutions[].id

## Read csv

In [1]:
import pandas as pd

In [2]:
df_raw = pd.read_csv("../data/OpenAlex/openalex_ai_papers_full.csv")
df_raw.sample(20)

,id,title,publication_year,language,cited_by_count,referenced_works_count,fwci,institutions_distinct_count,countries_distinct_count,publication_type,...,authors_count,citation_top_1_percent,citation_top_10_percent,cited_by_percentile_year_min,cited_by_percentile_year_max,keyword_count,primary_topic_score,first_year_citations,topic_id,topic_name
288132,https://openalex.org/W4200407836,Solving the sparse QUBO on multiple GPUs for S...,2021,en,7,19,0.6990,9,0,proceedings-article,...,9,False,False,89.0,99.0,10,0.9999,1,T10682,Quantum Computing Algorithms and Architecture
26838,https://openalex.org/W2998148686,Torus Driving Mechanism with the Ability to Br...,2019,en,0,0,0.0000,8,1,journal-article,...,8,False,False,NaN,NaN,6,0.6092,0,T10181,Natural Language Processing Techniques
271738,https://openalex.org/W4404800755,KL-FedDis: A federated learning approach with ...,2024,en,9,24,3.2317,6,1,journal-article,...,6,False,True,98.0,99.0,6,1.0000,9,T10764,Privacy-Preserving Technologies in Data
60003,https://openalex.org/W2807181957,TIRAMISU Methodology for Semi-Automated Interp...,2016,en,1,0,0.4285,2,0,NaN,...,2,False,False,90.0,94.0,8,0.6447,0,T10320,Neural Networks and Applications
43922,https://openalex.org/W4386566860,Towards Speech to Speech Machine Translation f...,2023,en,10,18,1.7457,2,1,proceedings-article,...,2,False,False,97.0,98.0,13,0.9998,6,T10181,Natural Language Processing Techniques
46104,https://openalex.org/W4321243131,"Орчин цагийн солонгос хэлийг монгол, турк хэлт...",2023,en,0,4,0.0000,1,0,journal-article,...,1,False,False,NaN,NaN,10,0.9900,0,T10181,Natural Language Processing Techniques
967,https://openalex.org/W2963455036,Modeling Compositionality with Multiplicative ...,2015,en,13,0,1.7897,2,1,NaN,...,2,False,False,89.0,97.0,13,0.9986,3,T10181,Natural Language Processing Techniques
60649,https://openalex.org/W2542246449,Bio-inspired intelligent sensory processing wi...,2016,en,0,0,0.0000,3,0,NaN,...,3,False,False,NaN,NaN,7,0.9538,0,T10320,Neural Networks and Applications
140824,https://openalex.org/W4405009677,Explainable Supporting Facts Generation via Qu...,2024,en,0,0,0.0000,2,0,journal-article,...,2,False,False,NaN,NaN,10,0.9976,0,T10028,Topic Modeling
93576,https://openalex.org/W2242198393,More A than I: Why Artificial Intelligence Isn...,2015,en,1,21,0.4314,1,1,proceedings-article,...,1,False,False,90.0,94.0,10,0.9949,0,T10028,Topic Modeling


In [3]:
print(f"Shape: {df_raw.shape}")
print(f"Years: {df_raw['publication_year'].min()} – {df_raw['publication_year'].max()}")
print(f"Topics: {df_raw['topic_name'].nunique()}")


Shape: (301406, 22)
Years: 2015 – 2024
Topics: 10


In [4]:
df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 301406 entries, 0 to 301405
Data columns (total 22 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   id                            301406 non-null  object 
 1   title                         301263 non-null  object 
 2   publication_year              301406 non-null  int64  
 3   language                      298519 non-null  object 
 4   cited_by_count                301406 non-null  int64  
 5   referenced_works_count        301406 non-null  int64  
 6   fwci                          295292 non-null  float64
 7   institutions_distinct_count   301406 non-null  int64  
 8   countries_distinct_count      301406 non-null  int64  
 9   publication_type              265754 non-null  object 
 10  is_oa                         301406 non-null  bool   
 11  oa_status                     301406 non-null  object 
 12  authors_count                 301406 non-nul

## Clean Null Values

In [5]:
df_raw.isnull().sum()

id                                  0
title                             143
publication_year                    0
language                         2887
cited_by_count                      0
referenced_works_count              0
fwci                             6114
institutions_distinct_count         0
countries_distinct_count            0
publication_type                35652
is_oa                               0
oa_status                           0
authors_count                       0
citation_top_1_percent           6114
citation_top_10_percent          6114
cited_by_percentile_year_min    94777
cited_by_percentile_year_max    94777
keyword_count                       0
primary_topic_score                 0
first_year_citations                0
topic_id                            0
topic_name                          0
dtype: int64

There are null values in columns 'title', 'language', 'fwci', 'publication_type', 'citation_top_1_percent', 'citation_top_10_percent', 'cited_by_percentile_year_min', 'cited_by_percentile_year_max'


Our target is 'citation_top_10_percent', so we can drop all rows where this column is null
But first let's check waht is the distribution of the rows with null in years and topics compared to the distribution of full data

In [6]:
df_raw[df_raw['citation_top_10_percent'].isnull()][['publication_year', 'topic_name']].value_counts().sort_index()

publication_year  topic_name                                    
2015              Anomaly Detection Techniques and Applications      19
                  Evolutionary Algorithms and Applications           17
                  Metaheuristic Optimization Algorithms Research      8
                  Natural Language Processing Techniques            109
                  Neural Networks and Applications                  107
                                                                   ... 
2024              Privacy-Preserving Technologies in Data            48
                  Quantum Computing Algorithms and Architecture      40
                  Sentiment Analysis and Opinion Mining              31
                  Speech Recognition and Synthesis                   17
                  Topic Modeling                                     48
Name: count, Length: 100, dtype: int64

Let's make it visual in the chart

In [7]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

null_target = df_raw[df_raw['citation_top_10_percent'].isnull()]

year_full  = df_raw['publication_year'].value_counts().sort_index()
year_null  = null_target['publication_year'].value_counts().sort_index()

topic_full = df_raw['topic_name'].value_counts()
topic_null = null_target['topic_name'].value_counts()

# Chart 1 — by year
fig_year = go.Figure()
fig_year.add_trace(go.Bar(name='Full dataset',    x=year_full.index.astype(str), y=year_full.values,  marker_color='steelblue'))
fig_year.add_trace(go.Bar(name='Null target rows', x=year_null.index.astype(str), y=year_null.values, marker_color='coral'))
fig_year.update_layout(barmode='group', title='Distribution by year', xaxis_title='Year', yaxis_title='Count', height=400)
fig_year.show()

# Chart 2 — by topic
fig_topic = go.Figure()
fig_topic.add_trace(go.Bar(name='Full dataset',    x=topic_full.values, y=topic_full.index, orientation='h', marker_color='steelblue'))
fig_topic.add_trace(go.Bar(name='Null target rows', x=topic_null.values, y=topic_null.index, orientation='h', marker_color='coral'))
fig_topic.update_layout(barmode='group', title='Distribution by topic', xaxis_title='Count', height=500)
fig_topic.show()

We can see that the distribution of the rows with null values in our target column 'citation_top_10_percent' is relatevily equal through the topics and years, so we can drop these rows.

In [8]:
df = df_raw.dropna(subset=['citation_top_10_percent']).copy()
print(f"Dropped {len(df_raw) - len(df):,} rows. Remaining: {len(df):,}")

Dropped 6,114 rows. Remaining: 295,292


Checking for null values after dropping the rows

In [9]:
df.isnull().sum()

id                                  0
title                             126
publication_year                    0
language                         2877
cited_by_count                      0
referenced_works_count              0
fwci                                0
institutions_distinct_count         0
countries_distinct_count            0
publication_type                34816
is_oa                               0
oa_status                           0
authors_count                       0
citation_top_1_percent              0
citation_top_10_percent             0
cited_by_percentile_year_min    88663
cited_by_percentile_year_max    88663
keyword_count                       0
primary_topic_score                 0
first_year_citations                0
topic_id                            0
topic_name                          0
dtype: int64

The biggest amount of nulls is in columns 'cited_by_percentile_year_min' and 'cited_by_percentile_year_max', it's almost 1/3 of the whole data.  
These columns are also not useful for the model bcause they are leaky and ill not be used as features.   
We can drop these two columns entirely, but first lets check their distribution through topics and years.

In [10]:
null_percentile = df[df['cited_by_percentile_year_min'].isnull() | df['cited_by_percentile_year_max'].isnull()]

year_full      = df['publication_year'].value_counts().sort_index()
year_null_perc = null_percentile['publication_year'].value_counts().sort_index()

topic_full      = df['topic_name'].value_counts()
topic_null_perc = null_percentile['topic_name'].value_counts()

# Chart 1 — by year
fig_year = go.Figure()
fig_year.add_trace(go.Bar(name='Full dataset',    x=year_full.index.astype(str),      y=year_full.values,      marker_color='steelblue'))
fig_year.add_trace(go.Bar(name='Null percentile', x=year_null_perc.index.astype(str), y=year_null_perc.values, marker_color='coral'))
fig_year.update_layout(barmode='group', title='Null in cited_by_percentile_year by year', xaxis_title='Year', yaxis_title='Count', height=400)
fig_year.show()

# Chart 2 — by topic
fig_topic = go.Figure()
fig_topic.add_trace(go.Bar(name='Full dataset',    x=topic_full.values,      y=topic_full.index,      orientation='h', marker_color='steelblue'))
fig_topic.add_trace(go.Bar(name='Null percentile', x=topic_null_perc.values, y=topic_null_perc.index, orientation='h', marker_color='coral'))
fig_topic.update_layout(barmode='group', title='Null in cited_by_percentile_year by topic', xaxis_title='Count', height=500)
fig_topic.show()

In [11]:
df['cited_by_percentile_year_min'].unique()

array([89., 97., 99., 90., 94., 96., 95., 98., 91., 93., nan])

In [12]:
df['cited_by_percentile_year_max'].unique()

array([100.,  99.,  98.,  97.,  96.,  95.,  94.,  93.,  nan])

The distribution of the null values is relatevelu equal - 25% of data in topics and years, this is a high volumne.  

We can drop these columns.

In [13]:
df_20 = df.drop(columns=['cited_by_percentile_year_min', 'cited_by_percentile_year_max'])
print(df_20.shape)


(295292, 20)


In [14]:
df_20.isnull().sum()

id                                 0
title                            126
publication_year                   0
language                        2877
cited_by_count                     0
referenced_works_count             0
fwci                               0
institutions_distinct_count        0
countries_distinct_count           0
publication_type               34816
is_oa                              0
oa_status                          0
authors_count                      0
citation_top_1_percent             0
citation_top_10_percent            0
keyword_count                      0
primary_topic_score                0
first_year_citations               0
topic_id                           0
topic_name                         0
dtype: int64

Now only 3 columns with nulls : tiltle 126, language 2877, publication_type 34816  

Let's check the distribution of missing publication_type through topics and years.

In [15]:
null_pub_type = df_20[df_20['publication_type'].isnull()]

year_full     = df_20['publication_year'].value_counts().sort_index()
year_null_pub = null_pub_type['publication_year'].value_counts().sort_index()

topic_full     = df_20['topic_name'].value_counts()
topic_null_pub = null_pub_type['topic_name'].value_counts()

# Chart 1 — by year
fig_year = go.Figure()
fig_year.add_trace(go.Bar(name='Full dataset',         x=year_full.index.astype(str),     y=year_full.values,     marker_color='steelblue'))
fig_year.add_trace(go.Bar(name='Null publication_type', x=year_null_pub.index.astype(str), y=year_null_pub.values, marker_color='coral'))
fig_year.update_layout(barmode='group', title='Null in publication_type by year', xaxis_title='Year', yaxis_title='Count', height=400)
fig_year.show()

# Chart 2 — by topic
fig_topic = go.Figure()
fig_topic.add_trace(go.Bar(name='Full dataset',         x=topic_full.values,     y=topic_full.index,     orientation='h', marker_color='steelblue'))
fig_topic.add_trace(go.Bar(name='Null publication_type', x=topic_null_pub.values, y=topic_null_pub.index, orientation='h', marker_color='coral'))
fig_topic.update_layout(barmode='group', title='Null in publication_type by topic', xaxis_title='Count', height=500)
fig_topic.show()

The missing values in publication_type are distributed rather equally through topics and years, except that is missing completely in years 2022-2024 which might indicate OpenAlex coverage issue for older years.  
Because this featue is valuable for the model - it indicates the type of article which influences the citation a lot - we will fill the nulls with 'unknown'

In [16]:
df_20['publication_type'] = df_20['publication_type'].fillna('unknown')

In [17]:
df_20.isnull().sum()

id                                0
title                           126
publication_year                  0
language                       2877
cited_by_count                    0
referenced_works_count            0
fwci                              0
institutions_distinct_count       0
countries_distinct_count          0
publication_type                  0
is_oa                             0
oa_status                         0
authors_count                     0
citation_top_1_percent            0
citation_top_10_percent           0
keyword_count                     0
primary_topic_score               0
first_year_citations              0
topic_id                          0
topic_name                        0
dtype: int64

Now we have two columns with nulls - title and language.  
Because this is a very little fraction of the whole data, we can fill those with 'unknown'

In [18]:
df_20['language'] = df_20['language'].fillna('unknown')
df_20['title']    = df_20['title'].fillna('unknown')

In [19]:
df_20.isnull().sum()

id                             0
title                          0
publication_year               0
language                       0
cited_by_count                 0
referenced_works_count         0
fwci                           0
institutions_distinct_count    0
countries_distinct_count       0
publication_type               0
is_oa                          0
oa_status                      0
authors_count                  0
citation_top_1_percent         0
citation_top_10_percent        0
keyword_count                  0
primary_topic_score            0
first_year_citations           0
topic_id                       0
topic_name                     0
dtype: int64

## Clean Duplicates

Let's check for duplicates

In [20]:
print(df_20.duplicated(subset=['id']).sum())

0


## Clean Datatypes

In [21]:
df_20.dtypes

id                              object
title                           object
publication_year                 int64
language                        object
cited_by_count                   int64
referenced_works_count           int64
fwci                           float64
institutions_distinct_count      int64
countries_distinct_count         int64
publication_type                object
is_oa                             bool
oa_status                       object
authors_count                    int64
citation_top_1_percent          object
citation_top_10_percent         object
keyword_count                    int64
primary_topic_score            float64
first_year_citations             int64
topic_id                        object
topic_name                      object
dtype: object

'citation_top_1_percent' and 'citation_top_10_percent' — currently object. These need to be converted to integers for the model, to create target variable.

'is_oa' — is bool, should be converted to int (0/1) for the model.

In [22]:
df_20['citation_top_1_percent']  = df_20['citation_top_1_percent'].map({'True': 1, 'False': 0, True: 1, False: 0}).astype('Int64')
df_20['citation_top_10_percent'] = df_20['citation_top_10_percent'].map({'True': 1, 'False': 0, True: 1, False: 0}).astype('Int64')
df_20['is_oa']                   = df_20['is_oa'].astype(int)

print(df_20.dtypes)

id                              object
title                           object
publication_year                 int64
language                        object
cited_by_count                   int64
referenced_works_count           int64
fwci                           float64
institutions_distinct_count      int64
countries_distinct_count         int64
publication_type                object
is_oa                            int64
oa_status                       object
authors_count                    int64
citation_top_1_percent           Int64
citation_top_10_percent          Int64
keyword_count                    int64
primary_topic_score            float64
first_year_citations             int64
topic_id                        object
topic_name                      object
dtype: object


In [23]:
df_20 = df_20.reset_index(drop=True)

In [24]:
df_20.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 295292 entries, 0 to 295291
Data columns (total 20 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   id                           295292 non-null  object 
 1   title                        295292 non-null  object 
 2   publication_year             295292 non-null  int64  
 3   language                     295292 non-null  object 
 4   cited_by_count               295292 non-null  int64  
 5   referenced_works_count       295292 non-null  int64  
 6   fwci                         295292 non-null  float64
 7   institutions_distinct_count  295292 non-null  int64  
 8   countries_distinct_count     295292 non-null  int64  
 9   publication_type             295292 non-null  object 
 10  is_oa                        295292 non-null  int64  
 11  oa_status                    295292 non-null  object 
 12  authors_count                295292 non-null  int64  
 13 

## Checking unique values

In [25]:
df_20.columns.to_list()

['id',
 'title',
 'publication_year',
 'language',
 'cited_by_count',
 'referenced_works_count',
 'fwci',
 'institutions_distinct_count',
 'countries_distinct_count',
 'publication_type',
 'is_oa',
 'oa_status',
 'authors_count',
 'citation_top_1_percent',
 'citation_top_10_percent',
 'keyword_count',
 'primary_topic_score',
 'first_year_citations',
 'topic_id',
 'topic_name']

In [26]:
for col in df_20.columns:
    print(f"\n{col} ({df_20[col].nunique()} unique):")
    print(df_20[col].value_counts().head(20))
    print("-" * 50)


id (295292 unique):
id
https://openalex.org/W7134285326    1
https://openalex.org/W1902237438    1
https://openalex.org/W658020064     1
https://openalex.org/W2251135946    1
https://openalex.org/W1491002046    1
https://openalex.org/W2250342921    1
https://openalex.org/W2100664567    1
https://openalex.org/W1847618513    1
https://openalex.org/W1503259811    1
https://openalex.org/W2118434577    1
https://openalex.org/W2251743902    1
https://openalex.org/W2251329024    1
https://openalex.org/W2134036914    1
https://openalex.org/W2251622960    1
https://openalex.org/W2133458109    1
https://openalex.org/W2114719613    1
https://openalex.org/W3204406378    1
https://openalex.org/W2294774419    1
https://openalex.org/W1570098300    1
https://openalex.org/W2251324968    1
Name: count, dtype: int64
--------------------------------------------------

title (290624 unique):
title
unknown                                                                              126
IEEE Transactions on

Looking at the data what we see:

title has many junk records
There are IEEE "publication information", "Table of contents", "Front Matter" type titles — these are not research papers, they're journal metadata records. Worth dropping.

publication_type — 579 unique values is a mess
What should be a clean categorical has fragmented into many variants of the same thing. Needs consolidating into a few clean categories.

countries_distinct_count — 76,004 rows with 0 countries
That's a lot. Worth investigating — likely papers where OpenAlex couldn't resolve any institution to a country.

authors_count — mirrors institutions_distinct_count exactly
Suspicious — they have identical value counts. Could be a data pull issue worth checking.

referenced_works_count — 74,790 rows with 0
That's 25% of papers with no references. Could be legitimate (some paper types don't have structured references in OpenAlex) but worth a quick look.

## Cleaning titles

### Duplicated titles


Let's check the duplicated rows in title, year, fwci, cited by count

In [27]:
duplicate_titles = df_20[df_20.duplicated(subset=['title'], keep=False)].sort_values('title')
print(f"Rows with duplicate titles: {len(duplicate_titles):,}")
print(f"Unique titles that repeat: {duplicate_titles['title'].nunique():,}")
duplicate_titles[['title', 'cited_by_count', 'fwci', 'publication_year', 'topic_name']]

Rows with duplicate titles: 7,858
Unique titles that repeat: 3,190


,title,cited_by_count,fwci,publication_year,topic_name
255718,&lt;i&gt;K&lt;/i&gt;-anonymity scheme for priv...,0,0.0000,2021,Privacy-Preserving Technologies in Data
255256,&lt;i&gt;K&lt;/i&gt;-anonymity scheme for priv...,4,0.4079,2021,Privacy-Preserving Technologies in Data
73541,(Invited) Binary Stochastic Neurons and Compou...,0,0.0000,2020,Neural Networks and Applications
78037,(Invited) Binary Stochastic Neurons and Compou...,0,0.0000,2021,Neural Networks and Applications
4902,12th Congress for Finno-Ugric Studies: Persona...,0,0.0000,2015,Natural Language Processing Techniques
...,...,...,...,...,...
227949,配電系統状態推定へのDifferential Evolutionary Particle S...,0,0.0000,2017,Metaheuristic Optimization Algorithms Research
234331,학술·연구정보가이드: Computer Science 분야 (03): Algorith...,0,0.0000,2020,Metaheuristic Optimization Algorithms Research
234546,학술·연구정보가이드: Computer Science 분야 (03): Algorith...,0,0.0000,2020,Metaheuristic Optimization Algorithms Research
74420,학술·연구정보가이드: Computer Science 분야 (03): Gaussian...,0,0.0000,2020,Neural Networks and Applications


Let's check what titles have same year, citation amount, fwci

In [28]:
exact_dupes = df_20.duplicated(subset=['title', 'publication_year', 'cited_by_count', 'fwci', 'topic_name'], keep=False)
print(f"Exact duplicates (title + year + citations): {exact_dupes.sum():,}")

Exact duplicates (title + year + citations): 2,932


In [29]:
df_20[exact_dupes].sort_values('title')[['id','title','publication_year', 'cited_by_count', 'fwci', 'topic_name']].head(10)

,id,title,publication_year,cited_by_count,fwci,topic_name
5300,https://openalex.org/W4300009002,12th Congress for Finno-Ugric Studies: Persona...,2015,0,0.0,Natural Language Processing Techniques
4902,https://openalex.org/W2987809845,12th Congress for Finno-Ugric Studies: Persona...,2015,0,0.0,Natural Language Processing Techniques
27090,https://openalex.org/W3197412964,13. Ulusal Yazılım Mühendisliği Sempozyumu,2019,0,0.0,Natural Language Processing Techniques
26347,https://openalex.org/W3006520634,13. Ulusal Yazılım Mühendisliği Sempozyumu,2019,0,0.0,Natural Language Processing Techniques
234605,https://openalex.org/W4251598402,2020 4th Conference on Swarm Intelligence and ...,2020,0,0.0,Metaheuristic Optimization Algorithms Research
234607,https://openalex.org/W4252153203,2020 4th Conference on Swarm Intelligence and ...,2020,0,0.0,Metaheuristic Optimization Algorithms Research
79941,https://openalex.org/W4286383371,3:1M: An Interpretable Dynamical AI Agent,2022,0,0.0,Neural Networks and Applications
80119,https://openalex.org/W4304459472,3:1M: An Interpretable Dynamical AI Agent,2022,0,0.0,Neural Networks and Applications
80123,https://openalex.org/W4306195409,3:1M: An Interpretable Dynamical AI Agent,2022,0,0.0,Neural Networks and Applications
79937,https://openalex.org/W4285832920,3:1M: An Interpretable Dynamical AI Agent,2022,0,0.0,Neural Networks and Applications


Dropping those rows with duplicated titles

In [30]:
df_20 = df_20.drop_duplicates(subset=['title', 'publication_year', 'cited_by_count'], keep='first').reset_index(drop=True)
print(f"Remaining rows: {len(df_20):,}")

Remaining rows: 293,298


### Junk titles

In [31]:
print(f"title ({df_20['title'].nunique()} unique):")
print(df_20['title'].value_counts().head(50))

title (290624 unique):
title
unknown                                                                                                                  91
IEEE Journal of Selected Topics in Quantum Electronics                                                                   29
Poster                                                                                                                   13
IEEE Journal of Selected Topics in Quantum Electronics Topic Codes and Topics                                            13
Preface                                                                                                                  11
IEEE Transactions on Information Theory publication information                                                          10
Sentiment Analysis of Twitter Data                                                                                        9
IEEE Transactions on Evolutionary Computation Society Information                                      

There are many strange titles that are probably not articles, let's try to catch them and drop
Such as 'Front Matter', 'Table of Contents'etc

In [32]:
junk_titles = [
    'Preface', 'Front Matter', 'Table of contents', 'Table of Contents',
    'information for authors',
    'Sumário Bilingue', 'Sumário bilingue', 'Summaries in Tamil', 'Summaries',
    'Foreword', 'Keynote', 'Tutorials', 'Final Program',
    'Message from the Chairs', 'Peer Review Statement', 'Book Review',
    'Book review', 'Editorial introduction',
    '[25 Years Ago]', 'Suunvuoro',
    'Table of contents in English',
    'ICAI-CMA Snapshots',
    'Hyper-heuristics tutorial', 'Summaries in Tamil',
    'Preface to the Special Issue on Theory of Genetic and Evolutionary Computation',
    'Announcement of the Neural Networks Best Paper Award',
    'French language abstracts', 'Editorial'
]

junk_patterns = [
    'publication information', 'Information for Authors',
    'Society Information',
    ' Edics$'  # only match Edics at end of title
]

mask_exact   = df_20['title'].isin(junk_titles)
mask_pattern = df_20['title'].str.contains('|'.join(junk_patterns), case=False, na=False, regex=True)

print(f"Exact matches:   {mask_exact.sum():,}")
print(f"Pattern matches: {mask_pattern.sum():,}")
print(f"Total to drop:   {(mask_exact | mask_pattern).sum():,}")

# Verify no false positives
df_20[mask_exact | mask_pattern][['title', 'cited_by_count', 'fwci']].sort_values('cited_by_count', ascending=False).head(20)

Exact matches:   112
Pattern matches: 116
Total to drop:   226


,title,cited_by_count,fwci
168958,Editorial,11,1.224400
287317,IEEE Journal of Selected Topics in Quantum Ele...,3,0.525400
75625,Announcement of the Neural Networks Best Paper...,2,0.271900
245189,Keynote,2,0.390000
64348,IEEE TRANSACTIONS ON NEURAL NETWORKS AND LEARN...,2,0.325800
75433,IEEE TRANSACTIONS ON NEURAL NETWORKS AND LEARN...,2,0.141104
58222,IEEE TRANSACTIONS ON NEURAL NETWORKS AND LEARN...,1,0.428500
58223,IEEE Transactions on Neural Networks informati...,1,0.000000
72069,IEEE TRANSACTIONS ON NEURAL NETWORKS AND LEARN...,1,0.000000
58225,IEEE Transactions on Signal Processing Edics,1,0.428500


In [33]:
with pd.option_context('display.max_rows', None, 'display.max_colwidth', None):
    print(df_20[mask_exact | mask_pattern][['title', 'cited_by_count', 'fwci', 'publication_year', 'topic_name']].sort_values('cited_by_count', ascending=False).to_string())

                                                                                                title  cited_by_count      fwci  publication_year                                      topic_name
168958                                                                                      Editorial              11  1.224400              2020           Sentiment Analysis and Opinion Mining
287317                 IEEE Journal of Selected Topics in Quantum Electronics Information for Authors               3  0.525400              2023   Quantum Computing Algorithms and Architecture
75625                                            Announcement of the Neural Networks Best Paper Award               2  0.271900              2021                Neural Networks and Applications
245189                                                                                        Keynote               2  0.390000              2017         Privacy-Preserving Technologies in Data
64348               IEEE TRANS

Let's drop the rows with the junk titles

In [34]:
df_20 = df_20[~(mask_exact | mask_pattern)].reset_index(drop=True)
print(f"Remaining rows: {len(df_20):,}")

Remaining rows: 293,072


In [35]:
# Discovered during EDA — IEEE index documents have unrealistic institutions_distinct_count (up to 4787) and 0 citations
# Drop IEEE index documents
ieee_index_mask = df_20['title'].str.contains('Index IEEE|Index IEEE/ACM', case=False, na=False)
print(f"IEEE index papers to drop: {ieee_index_mask.sum():,}")
df_20 = df_20[~ieee_index_mask].reset_index(drop=True)
print(f"Remaining rows: {len(df_20):,}")

IEEE index papers to drop: 19
Remaining rows: 293,053


In [36]:
# Discovered durind EDA
# Drop NeurIPS proceedings volumes — these are entire conference proceedings volumes
# misidentified by OpenAlex as individual papers, accumulating citations from all 
# papers within the volume. Not individual research papers.
neurips_mask = df_20['title'].str.contains('Advances in Neural Information Processing Systems', case=False, na=False)
print(f"NeurIPS proceedings to drop: {neurips_mask.sum()}")
df_20 = df_20[~neurips_mask].reset_index(drop=True)
print(f"Remaining rows: {len(df_20):,}")

NeurIPS proceedings to drop: 4
Remaining rows: 293,049


In [37]:
# Discovered durind EDA
# Drop CNS annual meeting abstract collections — published as single journal articles
# but contain abstracts from hundreds of presenters, accumulating their citations.
# Not individual research papers.
cns_mask = df_20['title'].str.contains('Annual Computational Neuroscience Meeting', case=False, na=False)
print(f"CNS meeting abstracts to drop: {cns_mask.sum()}")
df_20 = df_20[~cns_mask].reset_index(drop=True)
print(f"Remaining rows: {len(df_20):,}")

CNS meeting abstracts to drop: 2
Remaining rows: 293,047


In [38]:
# Discovered during EDA
# Drop non-research paper records:
# - 'Proceedings of the Second Workshop on Arabic Natural Language Processing' — workshop
#   proceedings collection misclassified as journal article, 53 authors, 0 citations
# - 'Universal Dependencies 2.7' — linguistic dataset/resource release, not a research paper
non_paper_mask = df_20['title'].isin([
    'Proceedings of the Second Workshop on Arabic Natural Language Processing',
    'Universal Dependencies 2.7'
])
print(f"Non-research paper records to drop: {non_paper_mask.sum()}")
df_20 = df_20[~non_paper_mask].reset_index(drop=True)
print(f"Remaining rows: {len(df_20):,}")

Non-research paper records to drop: 2
Remaining rows: 293,045


## Cleaning publication_type

In [39]:
print(f"publication_type ({df_20['publication_type'].nunique()} unique):")
print(df_20['publication_type'].value_counts().to_string())

publication_type (579 unique):


publication_type
journal-article                                                                                                                                                                                                                                                                    127238
proceedings-article                                                                                                                                                                                                                                                                112417
unknown                                                                                                                                                                                                                                                                             34647
text                                                                                                                                     

There are 579 different publication types, which is a mess, so let's classify them into 5 groups: journal article, proceedings article, thesis, other, unknown

In [40]:
def map_publication_type(val):
    val = str(val).lower().strip()
    
    journal_keywords = ['journal', 'article', 'text', 'periodical', 'peer review', 
                        'zeitschrift', 'artículo', 'artikel', 'artykuł', 'rivista']
    conference_keywords = ['conference', 'proceedings', 'workshop', 'konferenz', 
                           'inproceedings', 'confer', 'congres', 'kongres']
    thesis_keywords = ['thesis', 'dissertation', 'doctoral', 'master', 'bachelor', 
                       'phd', 'licentiate', 'etd', 'tese', 'thèse', 'disertac']
    
    if val == 'unknown':
        return 'unknown'
    for kw in conference_keywords:
        if kw in val:
            return 'proceedings-article'
    for kw in thesis_keywords:
        if kw in val:
            return 'thesis'
    for kw in journal_keywords:
        if kw in val:
            return 'journal-article'
    return 'other'

df_20['publication_type'] = df_20['publication_type'].apply(map_publication_type)
print(df_20['publication_type'].value_counts())

publication_type
journal-article        136228
proceedings-article    116167
unknown                 34647
other                    4378
thesis                   1625
Name: count, dtype: int64


## Cleaning countries_distinct_counts

Let's see where the articles with 0 countries are concentrated

In [41]:
zeros = df_20[df_20['countries_distinct_count'] == 0]
print(f"Rows with 0 countries: {len(zeros):,}")
print(f"\nBy publication_type:")
print(zeros['publication_type'].value_counts())
print(f"\nBy publication_year:")
print(zeros['publication_year'].value_counts().sort_index())
print(f"\nBy topic_name:")
print(zeros['topic_name'].value_counts())

Rows with 0 countries: 74,428

By publication_type:
publication_type
unknown                26573
journal-article        25510
proceedings-article    17944
other                   2841
thesis                  1560
Name: count, dtype: int64

By publication_year:
publication_year
2015    7129
2016    8024
2017    7295
2018    8270
2019    9504
2020    8980
2021    6559
2022    4116
2023    4778
2024    9773
Name: count, dtype: int64

By topic_name:
topic_name
Natural Language Processing Techniques            21610
Neural Networks and Applications                  13448
Topic Modeling                                     8249
Speech Recognition and Synthesis                   6381
Anomaly Detection Techniques and Applications      5539
Sentiment Analysis and Opinion Mining              5514
Quantum Computing Algorithms and Architecture      5108
Privacy-Preserving Technologies in Data            3918
Metaheuristic Optimization Algorithms Research     2818
Evolutionary Algorithms and Applic

There are 25% of articles with 0 country value, which may indicate the OpenAlex data entry issue. Let's look at this later when we pull more data about institutions

## Cleaning authors_count / institutions_distinct_count

As we noticed before, the authors count was almost reflecting the institutions count, let's see how many rows have actually equal count

In [42]:
print("authors_count == institutions_distinct_count:")
print((df_20['authors_count'] == df_20['institutions_distinct_count']).sum())
print(f"\nTotal rows: {len(df_20):,}")

print("\nCorrelation:")
print(df_20['authors_count'].corr(df_20['institutions_distinct_count']))

authors_count == institutions_distinct_count:
292960

Total rows: 293,045

Correlation:
0.861784030623429


In [43]:
293005-292918

87

Only 87 rows are different in authors count, let's see why

In [44]:
print(df_20[['authors_count', 'institutions_distinct_count']].describe())
print("\nSample where they differ:")
print(df_20[df_20['authors_count'] != df_20['institutions_distinct_count']][['authors_count', 'institutions_distinct_count']].head(20))

       authors_count  institutions_distinct_count
count  293045.000000                293045.000000
mean        3.543456                     3.560429
std         3.485310                     4.589382
min         0.000000                     0.000000
25%         2.000000                     2.000000
50%         3.000000                     3.000000
75%         4.000000                     4.000000
max       100.000000                  1229.000000

Sample where they differ:
       authors_count  institutions_distinct_count
5240             100                          101
13316            100                          142
27209            100                          131
32663            100                          227
35696            100                          108
39057            100                          115
41108            100                          114
61134            100                          121
63985            100                          132
67113            100   

In [45]:
print(df_20['authors_count'].value_counts().sort_index().tail(20))

authors_count
81      2
82      7
83      4
84      5
85      2
86      6
87      5
88      1
89      4
90      2
91      2
92      4
93      2
94      3
95      6
96      7
97      2
98      1
99      3
100    85
Name: count, dtype: int64


So we see that the max value in authour count is 100, while in institution count its much higher - 1229.  
This might indicate that OpenAlex set a cap on authors count at 100, or that some authors represent more institutions.  
The difference group is very small - 87 items only.   


In [46]:
with pd.option_context('display.max_rows', None, 'display.max_colwidth', None):
    print(df_20[df_20['authors_count'] == 100][[
        'title', 'authors_count', 'institutions_distinct_count', 
        'countries_distinct_count', 'cited_by_count', 'publication_year', 
        'topic_name']].to_string())

                                                                                                                                                                                                             title  authors_count  institutions_distinct_count  countries_distinct_count  cited_by_count  publication_year                                      topic_name
5240                                                                                                                                               Constituency Parse Reranking for Morphologically Rich Languages            100                          101                         1               0              2015          Natural Language Processing Techniques
13316                                                                                                                                                              Evolving Fuzzy Models for Automated Translation            100                          142                    

In [47]:
for idx in [32662, 201653, 198648, 176507, 28430]:
    print(f"\nRow {idx}:")
    print(df_20.loc[idx, 'id'])


Row 32662:
https://openalex.org/W4212932563

Row 201653:
https://openalex.org/W4311213197

Row 198648:
https://openalex.org/W4213101961

Row 176507:
https://openalex.org/W4312383299

Row 28430:
https://openalex.org/W3176687562


After checking the difference between authors_count and institutions_distinct_count through the real links, it became clear that there is indeed a cap on authors_count and institutions_distinct_count represent the actual authors count.
Therefore we will drop the authors_count column and rename `institutions_distinct_count` into `authorship_count`

In [48]:
df_20 = df_20.drop(columns=['authors_count'])

In [49]:
df_20 = df_20.rename(columns={'institutions_distinct_count': 'authorship_count'})
df_20.columns.tolist()

['id',
 'title',
 'publication_year',
 'language',
 'cited_by_count',
 'referenced_works_count',
 'fwci',
 'authorship_count',
 'countries_distinct_count',
 'publication_type',
 'is_oa',
 'oa_status',
 'citation_top_1_percent',
 'citation_top_10_percent',
 'keyword_count',
 'primary_topic_score',
 'first_year_citations',
 'topic_id',
 'topic_name']

## Cleaning referenced_works_count

In [50]:
zeros_ref = df_20[df_20['referenced_works_count'] == 0]
print(f"Rows with 0 references: {len(zeros_ref):,}")
print(f"\nBy publication_type:")
print(zeros_ref['publication_type'].value_counts())
print(f"\nBy publication_year:")
print(zeros_ref['publication_year'].value_counts().sort_index())
print(f"\nBy topic_name:")
print(zeros_ref['topic_name'].value_counts())

Rows with 0 references: 72,778

By publication_type:


publication_type
journal-article        25583
unknown                25239
proceedings-article    17368
other                   3294
thesis                  1294
Name: count, dtype: int64

By publication_year:
publication_year
2015     6128
2016     6991
2017     6895
2018     8211
2019     8824
2020     8184
2021     6135
2022     4341
2023     4577
2024    12492
Name: count, dtype: int64

By topic_name:
topic_name
Natural Language Processing Techniques            21533
Neural Networks and Applications                  16022
Topic Modeling                                     5733
Anomaly Detection Techniques and Applications      5504
Speech Recognition and Synthesis                   5478
Quantum Computing Algorithms and Architecture      5070
Sentiment Analysis and Opinion Mining              4313
Privacy-Preserving Technologies in Data            4029
Metaheuristic Optimization Algorithms Research     2970
Evolutionary Algorithms and Applications           2126
Name: count, dtype: 

There are 25% of articles with 0 referenced_works value, which may indicate the OpenAlex data entry issue. Let's look at this later when we pull more data

## Save cleaned csv

In [51]:
df_20.to_csv('../data/OpenAlex/openalex_ai_papers_cleaned.csv', index=False)
print(f"Saved: {df_20.shape}")

Saved: (293045, 19)


In [52]:
df_clean = pd.read_csv("../data/OpenAlex/openalex_ai_papers_cleaned.csv")
df_clean.head()

,id,title,publication_year,language,cited_by_count,referenced_works_count,fwci,authorship_count,countries_distinct_count,publication_type,is_oa,oa_status,citation_top_1_percent,citation_top_10_percent,keyword_count,primary_topic_score,first_year_citations,topic_id,topic_name
0,https://openalex.org/W1902237438,Effective Approaches to Attention-based Neural...,2015,en,8508,20,633.3691,3,1,proceedings-article,1,gold,1,1,7,0.9999,155,T10181,Natural Language Processing Techniques
1,https://openalex.org/W658020064,From Word Embeddings To Document Distances,2015,en,1518,47,164.8113,4,1,other,0,closed,1,1,11,0.9996,61,T10181,Natural Language Processing Techniques
2,https://openalex.org/W2251135946,Distant Supervision for Relation Extraction vi...,2015,en,1200,31,87.4904,4,1,proceedings-article,1,gold,1,1,11,0.9992,17,T10181,Natural Language Processing Techniques
3,https://openalex.org/W1491002046,An Introduction to Recursive Partitioning Usin...,2015,en,1030,7,105.2714,2,0,unknown,0,closed,1,1,14,0.9880,54,T10181,Natural Language Processing Techniques
4,https://openalex.org/W2250342921,chrF: character n-gram F-score for automatic M...,2015,en,935,10,36.6006,1,1,proceedings-article,1,gold,1,1,8,1.0000,14,T10181,Natural Language Processing Techniques
